# 02 — Model Training (Champion Model)

In [4]:
import pandas as pd

df = pd.read_csv("data/creditcard.csv")
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0



## Model Benchmarking (Logistic Regression vs Random Forest vs LightGBM)

In [2]:
from sklearn.model_selection import train_test_split

X = df.drop("Class", axis=1)
y = df["Class"]

# 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)

# temp -> 15% val, 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)
print("Fraud rate (train/val/test):", y_train.mean(), y_val.mean(), y_test.mean())

Train: (199364, 30) Val: (42721, 30) Test: (42722, 30)
Fraud rate (train/val/test): 0.0017254870488152324 0.0017321691907960957 0.0017321286456626562


## Train All The Three Models

In [3]:
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb

# Logistic Regression (interpretable baseline)
log_model = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42))
])
log_model.fit(X_train, y_train)
log_val_probs = log_model.predict_proba(X_val)[:, 1]

# Random Forest (nonlinear benchmark)
rf_model = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)
rf_val_probs = rf_model.predict_proba(X_val)[:, 1]

# LightGBM (scalable gradient boosting)
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

lgb_model = lgb.LGBMClassifier(
    n_estimators=1500,
    learning_rate=0.02,
    num_leaves=64,
    min_child_samples=50,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    scale_pos_weight= 577,
    random_state=42
)
lgb_model.fit(X_train, y_train)
lgb_val_probs = lgb_model.predict_proba(X_val)[:, 1]

print("Models trained.")

[LightGBM] [Info] Number of positive: 344, number of negative: 199020
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.143087 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7650
[LightGBM] [Info] Number of data points in the train set: 199364, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.001725 -> initscore=-6.360519
[LightGBM] [Info] Start training from score -6.360519
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gai

## Evaluate The Models 

In [4]:
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
import pandas as pd

MAX_FPR = 0.002  # 0.2%

def recall_at_fpr(y_true, probs, max_fpr=MAX_FPR):
    fpr, tpr, thresholds = roc_curve(y_true, probs)
    idx = np.where(fpr <= max_fpr)[0]
    if len(idx) == 0:
        return 0.0
    return float(tpr[idx[-1]])

def evaluate(name, y_true, probs):
    return {
        "Model": name,
        "ROC-AUC": roc_auc_score(y_true, probs),
        "PR-AUC": average_precision_score(y_true, probs),
        "Recall @0.2% FPR": recall_at_fpr(y_true, probs)
    }

results = [
    evaluate("Logistic Regression", y_val, log_val_probs),
    evaluate("Random Forest", y_val, rf_val_probs),
    evaluate("LightGBM", y_val, lgb_val_probs),
]

comparison_df = pd.DataFrame(results)
comparison_df

,Model,ROC-AUC,PR-AUC,Recall @0.2% FPR
0,Logistic Regression,0.968390,0.630087,0.824324
1,Random Forest,0.949292,0.809591,0.837838
2,LightGBM,0.923751,0.830157,0.837838


## Model Benchmarking Interpretation and Champion Selection

Three candidate models were evaluated using a stratified 70/15/15 split:

- Logistic Regression (baseline, interpretable model)
- Random Forest (non-linear ensemble benchmark)
- LightGBM (gradient boosting, scalable model)

### Evaluation Criteria

Given the highly imbalanced nature of fraud detection, model comparison prioritized:

1. PR-AUC (Precision-Recall AUC) — more informative than ROC-AUC for rare events  
2. Recall at fixed FPR ≤ 0.2% — aligned with operational fraud policy constraints  

### Results Summary

- Logistic Regression achieved the highest ROC-AUC but lower PR-AUC.
- Random Forest delivered strong PR-AUC and recall under constraint.
- Tuned LightGBM achieved the highest PR-AUC and matched the highest recall at the 0.2% FPR constraint.

### Champion Selection

LightGBM was selected as the validation champion because:

- It achieved the highest PR-AUC (strongest fraud ranking quality).
- It matched the best recall under operational constraint.
- It offers better scalability and production efficiency for large transaction volumes.

The model is now frozen and will be evaluated once on the untouched test set.

In [13]:
# Generate test probabilities using frozen LightGBM model
lgb_test_probs = lgb_model.predict_proba(X_test)[:, 1]

In [14]:
from sklearn.metrics import roc_curve
import numpy as np

MAX_FPR = 0.002  # 0.2%

fpr, tpr, thresholds = roc_curve(y_val, lgb_val_probs)

valid_idx = np.where(fpr <= MAX_FPR)[0]
best_idx = valid_idx[-1]

lgb_best_threshold = thresholds[best_idx]
val_recall = tpr[best_idx]
val_fpr = fpr[best_idx]

print("Frozen Validation Threshold:", lgb_best_threshold)
print("Validation Recall @0.2% FPR:", round(val_recall, 4))
print("Validation FPR:", round(val_fpr, 5))

Frozen Validation Threshold: 0.014554874923771423
Validation Recall @0.2% FPR: 0.8378
Validation FPR: 0.00054


## Validation Confusion Matrix (Frozen Threshold)

In [7]:
# Apply frozen threshold to test set
test_preds = (lgb_test_probs >= lgb_best_threshold).astype(int)

from sklearn.metrics import confusion_matrix, roc_auc_score, average_precision_score

tn, fp, fn, tp = confusion_matrix(y_test, test_preds).ravel()

test_roc = roc_auc_score(y_test, lgb_test_probs)
test_pr = average_precision_score(y_test, lgb_test_probs)

recall_test = tp / (tp + fn)
fpr_test = fp / (fp + tn)

print("TEST RESULTS")
print("ROC-AUC:", round(test_roc, 4))
print("PR-AUC:", round(test_pr, 4))
print("Recall @ frozen threshold:", round(recall_test, 4))
print("FPR:", round(fpr_test, 5))
print("Confusion Matrix -> TP:", tp, "FP:", fp, "FN:", fn, "TN:", tn)

TEST RESULTS
ROC-AUC: 0.9655
PR-AUC: 0.8385
Recall @ frozen threshold: 0.8378
FPR: 0.00042
Confusion Matrix -> TP: 62 FP: 18 FN: 12 TN: 42630


In [8]:
from sklearn.metrics import confusion_matrix

val_preds = (lgb_val_probs >= lgb_best_threshold).astype(int)
tn, fp, fn, tp = confusion_matrix(y_val, val_preds).ravel()

print("VALIDATION CONFUSION MATRIX")
print("TP:", tp, "FP:", fp, "FN:", fn, "TN:", tn)

VALIDATION CONFUSION MATRIX
TP: 62 FP: 23 FN: 12 TN: 42624


## Freeze Champion Artifacts (Model + Threshold)

In [15]:
import os, json, joblib

os.makedirs("models", exist_ok=True)

joblib.dump(lgb_model, "models/lgbm_champion.pkl")

with open("models/lgbm_threshold.json", "w") as f:
    json.dump({"threshold": lgb_best_threshold, "max_fpr": MAX_FPR}, f)

print("Saved: models/lgbm_champion.pkl")
print("Saved: models/lgbm_threshold.json")

Saved: models/lgbm_champion.pkl
Saved: models/lgbm_threshold.json


In [2]:
import os
os.getcwd()

'C:\\Users\\emeka\\JupyterProjects\\Fraud_Detection_ML_System'

In [10]:
import shutil, os

src = r"C:\Users\emeka\Downloads\creditcard.csv"
dst_dir = r"C:\Users\emeka\JupyterProjects\Fraud_Detection_ML_System\data"
dst = os.path.join(dst_dir, "creditcard.csv")

os.makedirs(dst_dir, exist_ok=True)
shutil.copyfile(src, dst)

print("Copied to:", dst)
print("Now in data/:", os.listdir("data"))

PermissionError: [Errno 13] Permission denied: 'C:\\Users\\emeka\\Downloads\\creditcard.csv'

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("data/creditcard.csv")

X = df.drop("Class", axis=1)
y = df["Class"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

sample_payload = X_val.iloc[0].to_dict()
sample_payload

{'Time': 164755.0,
 'V1': -0.7474005397269,
 'V2': 0.91181754726771,
 'V3': 0.912288991164576,
 'V4': -1.10489859025569,
 'V5': -0.295756587087449,
 'V6': -0.110130062688318,
 'V7': -0.111374388931085,
 'V8': 0.651699598130293,
 'V9': 0.299752561274583,
 'V10': -0.768238466456211,
 'V11': 0.311777121527008,
 'V12': 0.760186778618761,
 'V13': -0.240141125029152,
 'V14': 0.290112213826619,
 'V15': -0.334300220559032,
 'V16': 0.592012969725602,
 'V17': -0.736793804041992,
 'V18': 0.428382395498708,
 'V19': 0.174207757917884,
 'V20': -0.130816454357366,
 'V21': -0.121492943965705,
 'V22': -0.39937802159209,
 'V23': -0.0262164970339453,
 'V24': -0.461905685480406,
 'V25': -0.165569865544074,
 'V26': -0.256616980805932,
 'V27': 0.0063953426407206,
 'V28': 0.0014015780648457,
 'Amount': 0.77}

In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("data/creditcard.csv")

X = df.drop("Class", axis=1)
y = df["Class"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print("Split ready.")

Split ready.


In [10]:
import lightgbm as lgb

# compute scale_pos_weight automatically
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

lgb_model = lgb.LGBMClassifier(
    n_estimators=1500,
    learning_rate=0.02,
    num_leaves=64,
    min_child_samples=50,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    scale_pos_weight=577,
    random_state=42
)

lgb_model.fit(X_train, y_train)

lgb_val_probs = lgb_model.predict_proba(X_val)[:, 1]

print("LightGBM retrained.")

[LightGBM] [Info] Number of positive: 344, number of negative: 199020
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.071610 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7650
[LightGBM] [Info] Number of data points in the train set: 199364, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.001725 -> initscore=-6.360519
[LightGBM] [Info] Start training from score -6.360519
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gai

In [11]:
import numpy as np
from sklearn.metrics import roc_curve

MAX_FPR = 0.002  # 0.2%

fpr, tpr, thresholds = roc_curve(y_val, lgb_val_probs)

valid_idx = np.where(fpr <= MAX_FPR)[0]
best_idx = valid_idx[-1]

lgb_best_threshold = float(thresholds[best_idx])

print("Frozen validation threshold:", lgb_best_threshold)
print("Validation Recall:", round(float(tpr[best_idx]), 4))
print("Validation FPR:", round(float(fpr[best_idx]), 6))

Frozen validation threshold: 0.014554874923771423
Validation Recall: 0.8378
Validation FPR: 0.000539


In [12]:
import os
import json
import joblib

os.makedirs("models", exist_ok=True)

# Save model
joblib.dump(lgb_model, "models/lgbm_champion.pkl")

# Save threshold
with open("models/lgbm_threshold.json", "w") as f:
    json.dump(
        {
            "threshold": lgb_best_threshold,
            "max_fpr": MAX_FPR
        },
        f
    )

print("Model and threshold saved successfully.")

Model and threshold saved successfully.
